Criar uma tabela de produtos já unindo categoria,marca (brands)<br> e quantidade total  de estoque em todas as lojas

In [0]:
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
#criando um dicionario com o caminho de cada pasta do arquivo parquet/delta
bronze_map = {
    'tmp_brands': f'{bronze_path}brand/',
    'tmp_customers': f'{bronze_path}customers/',
    'tmp_orders': f'{bronze_path}orders/',
    'tmp_order_items': f'{bronze_path}orders_item/',
    'tmp_products': f'{bronze_path}products/',
    'tmp_stores': f'{bronze_path}stores/',
    'tmp_staff': f'{bronze_path}staffs/',
    'tmp_categories': f'{bronze_path}categories/',
    'tmp_stocks': f'{bronze_path}stocks/'
    }

#fazendo um looping para criar uma tabela temporaria pra cada tabela
for key, value in bronze_map.items():
    (spark.read.format('delta')
        .load(value)
        .createOrReplaceTempView(key)
)

In [0]:
#criando dataframe 

df_product = spark.sql("""
with
stock as ( 
select 
  product_id
  ,sum(quantity) as total_stock
from tmp_stocks
--where product_id = 1
group by product_id
)


select
   p.product_id
  ,p.product_name
  --,p.brand_id
  --,b.brand_id as brand_id_brand
  ,b.brand_name
  --,p.category_id
  --,c.category_id as category_id_categoriat
  ,c.category_name
  ,p.model_year
  ,p.list_price
  ,s.total_stock
from        tmp_products as p
left join tmp_categories as c on p.category_id = c.category_id
left join     tmp_brands as b on p.brand_id = b.brand_id
left join                 stock as s on p.product_id = s.product_id
""")



#salvar arquivo parquet na silver como delta
df_product.write\
.format('delta')\
.mode('overwrite')\
.option("mergeSchema", "true")\
.save(silver_path+'product')


In [0]:
#CRIANDO A TABELA
df = df_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.silver_product")

In [0]:
#CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_products
#OCATION 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/bikestore/silver/product';

#EU IA USAR ESSE COMANDO PRA CRIAR A TABELA VIA SQL MAS NAO DEU CERTO POIS O LOCATION TEM QUE SER EXTERNO